# 📖 Lab 2: Fault Tolerance — Pipeline Stages + Retry

**Non-functional requirement:** *Handle failures gracefully, resume without losing progress.*

Lab 1's monolithic crawler loses all progress on failure. The fix: **break it into pipelined
stages** so each stage can fail, retry, and be re-run independently.

## 🏗️ Architecture — Before vs After

```
❌ Before (Lab 1):   Queue ──> [fetch + extract + store] ──> single failure = all lost

✅ After (Pipeline):  Frontier ──> URL Fetcher ──> S3 Raw HTML ──> Parser ──> S3 Text
                      Queue         (retries)                      (retries)
                                    ↓ fail?                        ↓ fail?
                                    DLQ (after N receives)          retry independently
```

## Learning Objectives

- Split the monolithic crawler into fetch + parse stages
- Implement retry with exponential backoff — and watch a flaky URL *recover*
- Route permanently-dead URLs to a dead letter queue
- Re-run the parse stage over stored HTML **without re-fetching anything**


## 🛠️ Setup

```bash
cd 06-system-designs/web-crawler
docker compose up -d
```

Select the **"Python 3 (.venv)"** kernel.

In [ ]:
import http.server
import socket
import socketserver
import threading
import time
from collections import defaultdict
from dataclasses import dataclass
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup

print("✅ Ready.")

## 🖥️ A Server That Fails on Purpose

Retry logic is only worth studying if you can control *how* things fail. Crawling the real
web gives us "it worked" or "it didn't", never a reproducible sequence. So we run a local
origin server with three deliberately different failure modes:

| Path | Behaviour | What it teaches |
|------|-----------|-----------------|
| `/ok/<n>` | always 200 | the happy path |
| `/flaky/<n>` | 503 twice, then 200 | **backoff eventually succeeding** |
| `/broken/<n>` | always 503 | exhausting retries → DLQ |

Plus one URL pointing at a domain that does not resolve, for a DNS-level failure.

In [ ]:
# How many times each /flaky/ URL has been requested so far.
flaky_attempts: dict[str, int] = defaultdict(int)
FLAKY_FAILURES = 2  # fail this many times, then start succeeding

PAGE = ("<html><body><h1>{path}</h1><p>The main article body for {path}.</p>"
        "<a href='/ok/next'>next</a><a href='/ok/other'>other</a>"
        "<footer>Copyright notice, cookie banner, 40 nav links</footer>"
        "</body></html>")


class FlakyHandler(http.server.BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path.startswith("/broken/"):
            return self._send(503, "service unavailable")

        if self.path.startswith("/flaky/"):
            flaky_attempts[self.path] += 1
            if flaky_attempts[self.path] <= FLAKY_FAILURES:
                return self._send(503, "temporarily unavailable")

        self._send(200, PAGE.format(path=self.path))

    def _send(self, status: int, body: str):
        raw = body.encode()
        self.send_response(status)
        self.send_header("Content-Type", "text/html")
        self.send_header("Content-Length", str(len(raw)))
        self.end_headers()
        self.wfile.write(raw)

    def log_message(self, *args):
        pass


class ThreadedHTTPServer(socketserver.ThreadingMixIn, http.server.HTTPServer):
    daemon_threads = True


_probe = socket.socket()
_probe.bind(("127.0.0.1", 0))
PORT = _probe.getsockname()[1]
_probe.close()

origin_server = ThreadedHTTPServer(("127.0.0.1", PORT), FlakyHandler)
threading.Thread(target=origin_server.serve_forever, daemon=True).start()

ORIGIN = f"http://127.0.0.1:{PORT}"
print(f"✅ Flaky origin server on {ORIGIN}")

## 🔧 Simulating SQS: Visibility Timeout, Receive Count, DLQ

Three SQS features do all the work, and they are worth naming precisely:

- **Visibility timeout** — a received message is *hidden*, not deleted. If the worker
  crashes without deleting it, it reappears automatically. This is what makes the queue
  crash-safe rather than merely retry-safe.
- **`ChangeMessageVisibility`** — the worker extends the hidden period. That is how you
  implement backoff: don't sleep in the worker, push the message into the future and go
  do other work.
- **`maxReceiveCount` + DLQ** — after N deliveries the message is moved aside instead of
  being retried forever.

In [ ]:
@dataclass
class Message:
    url: str
    receive_count: int = 0
    visible_after: float = 0.0   # wall-clock time when this becomes receivable again


class SimulatedQueue:
    """SQS-like queue: visibility timeout, receive counting, dead letter queue."""

    def __init__(self, name: str, max_receives: int = 5):
        self.name = name
        self.messages: list[Message] = []
        self.max_receives = max_receives
        self.dlq: list[Message] = []

    def send(self, url: str) -> None:
        self.messages.append(Message(url=url))

    def receive(self) -> Message | None:
        """Return the first visible message, or None if all are hidden/exhausted."""
        now = time.time()
        # Iterate over a copy: messages can be moved to the DLQ mid-loop.
        for msg in list(self.messages):
            if msg.visible_after > now:
                continue
            if msg.receive_count >= self.max_receives:
                self.messages.remove(msg)
                self.dlq.append(msg)
                continue
            msg.receive_count += 1
            return msg
        return None

    def delete(self, msg: Message) -> None:
        """The worker succeeded — remove the message for good."""
        if msg in self.messages:
            self.messages.remove(msg)

    def change_visibility(self, msg: Message, timeout_seconds: float) -> None:
        """SQS ChangeMessageVisibility — hide this message for N more seconds."""
        msg.visible_after = time.time() + timeout_seconds

    def size(self) -> int:
        return len(self.messages)


print("✅ Queue defined.")

### The two stages

Storage stands in for S3 (raw HTML in one bucket, extracted text in another) and a
metadata DB that records where each artifact landed.

In [ ]:
raw_html: dict[str, str] = {}     # S3 bucket: raw HTML
text_store: dict[str, str] = {}   # S3 bucket: extracted text
metadata: dict[str, dict] = {}    # metadata DB: url -> {status, html_key, text_key}


def fetch(url: str) -> str | None:
    """Stage 1 — URL Fetcher. Returns HTML, or None on any failure."""
    try:
        resp = requests.get(url, timeout=5,
                            headers={"User-Agent": "EducationalCrawlerBot/1.0"})
        resp.raise_for_status()
        return resp.text
    except requests.RequestException:
        return None


DEFAULT_STRIP = ("script", "style", "nav", "footer")


def parse_html(html: str, url: str, strip_tags: tuple = DEFAULT_STRIP) -> dict:
    """Stage 2 — Parser. Extracts text + outbound URLs from *stored* HTML."""
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(list(strip_tags)):
        tag.decompose()

    urls = []
    for link in soup.find_all("a", href=True):
        parsed = urlparse(urljoin(url, link["href"]))
        if parsed.scheme in ("http", "https"):
            urls.append(f"{parsed.scheme}://{parsed.netloc}{parsed.path}")

    return {"text": soup.get_text(separator="\n", strip=True), "urls": sorted(set(urls))}


print("✅ Pipeline stages defined.")

## 🧪 Stage 1: Fetch with Exponential Backoff

Watch `/flaky/1`. It fails, gets hidden for a growing interval, and **succeeds on its
third delivery**. `/broken/1` and the bad DNS name never recover and end up in the DLQ.

We scale the backoff by 10x down (0.2s, 0.4s, 0.8s instead of 2s, 4s, 8s) so the lab
finishes in seconds. The growth curve is the point, not the absolute values.

In [ ]:
fetch_queue = SimulatedQueue("fetch_queue", max_receives=4)
parse_queue = SimulatedQueue("parse_queue", max_receives=3)

seeds = [
    f"{ORIGIN}/ok/1",                              # succeeds immediately
    f"{ORIGIN}/flaky/1",                           # 503, 503, then 200 → recovers
    f"{ORIGIN}/broken/1",                          # always 503 → DLQ
    "http://this-domain-does-not-exist-xyz.invalid/",  # DNS failure → DLQ
]
for url in seeds:
    fetch_queue.send(url)

BACKOFF_SCALE = 0.1   # demo speed-up: real backoff would be 2s, 4s, 8s...
DEADLINE = time.time() + 30   # safety valve so the lab can never hang

print("═" * 66)
print("  STAGE 1: URL Fetcher")
print("═" * 66)

while fetch_queue.size() > 0 and time.time() < DEADLINE:
    msg = fetch_queue.receive()
    if msg is None:
        # Everything is hidden by a backoff timer. A real worker would go poll
        # another queue instead of sleeping; we just wait for the next message.
        time.sleep(0.05)
        continue

    html = fetch(msg.url)
    label = msg.url.replace(ORIGIN, "")

    if html is not None:
        raw_html[msg.url] = html
        metadata[msg.url] = {"status": "fetched", "html_key": f"s3://raw/{label}"}
        parse_queue.send(msg.url)
        fetch_queue.delete(msg)
        print(f"  ✅ {label:<42} delivery {msg.receive_count} → {len(html)} bytes")
    else:
        backoff = 2 ** msg.receive_count      # 2s, 4s, 8s, ...
        fetch_queue.change_visibility(msg, backoff * BACKOFF_SCALE)
        print(f"  ❌ {label:<42} delivery {msg.receive_count}"
              f"/{fetch_queue.max_receives} → hidden for {backoff}s")

print(f"\n  📊 fetched: {len(raw_html)}   dead letter: {len(fetch_queue.dlq)}")
for msg in fetch_queue.dlq:
    print(f"     ☠️  {msg.url}  (failed {msg.receive_count} deliveries)")
print("\n  💡 /flaky/1 needed 3 deliveries and got there. A crawler without retry")
print("     would have thrown that page away on the first 503.")

## 🧪 Stage 2: Parse — and Re-parse

The parser never touches the network. It reads HTML that stage 1 already put in S3.

That separation is the real payoff, and the next cell proves it: we change the extraction
rules and re-run the parse stage over the **same stored HTML**, with the origin server
receiving zero additional requests. At 10B pages, a change to your text extractor is a
week of re-crawling if the stages are fused, and an afternoon of re-parsing if they
aren't.

In [ ]:
def run_parse_stage(queue: SimulatedQueue, label: str,
                    strip_tags: tuple = DEFAULT_STRIP) -> list[str]:
    discovered: list[str] = []
    print(f"  {label}")
    while True:
        msg = queue.receive()
        if msg is None:
            break
        result = parse_html(raw_html[msg.url], msg.url, strip_tags)
        text_store[msg.url] = result["text"]
        metadata[msg.url].update(status="parsed",
                                 text_key=f"s3://text{urlparse(msg.url).path}")
        discovered.extend(result["urls"])
        queue.delete(msg)
        print(f"    ✅ {msg.url.replace(ORIGIN, ''):<28} "
              f"{len(result['text']):>4} chars, {len(result['urls'])} URLs discovered")
    return discovered


print("═" * 66)
print("  STAGE 2: Text & URL Extraction")
print("═" * 66)
discovered = run_parse_stage(parse_queue, "first pass")

# ── Re-processing: the extraction logic changed, the crawl did not ──
requests_before = sum(flaky_attempts.values())

for url in raw_html:                       # re-enqueue everything we already have
    parse_queue.send(url)
print()
# New rule: keep <footer> text after all. Same bytes in S3, different text out.
_ = run_parse_stage(parse_queue,
                    "second pass: now keeping <footer> text — no re-fetching",
                    strip_tags=("script", "style", "nav"))

print(f"\n  Origin requests during re-parse: "
      f"{sum(flaky_attempts.values()) - requests_before}   ← zero, by design")

print(f"\n📊 Pipeline summary:")
print(f"  raw HTML stored   : {len(raw_html)}")
print(f"  text extracted    : {len(text_store)}")
print(f"  dead letter       : {len(fetch_queue.dlq)}")
print(f"  new URLs found    : {len(set(discovered))}")
print(f"\n  Metadata DB (resume-safe — a crash here loses nothing):")
for url, meta in metadata.items():
    print(f"    {url.replace(ORIGIN, ''):<14} {meta['status']:<8} {meta.get('text_key', '')}")

## 🧹 Cleanup

In [ ]:
origin_server.shutdown()
origin_server.server_close()
print("✅ Origin server stopped.")

## ✅ Summary

| Concept | How | What it costs |
|---------|-----|---------------|
| **Pipeline stages** | Fetch → S3 raw HTML → Parse → S3 text | you now store the raw HTML too — at 10B × 2MB that is 20 PB of bytes you would otherwise have thrown away |
| **Visibility timeout** | message hidden, not deleted, while a worker holds it | a crashed worker's URL is delayed by the timeout, not lost |
| **Exponential backoff** | `2^delivery` seconds via `ChangeMessageVisibility` | a genuinely slow site holds a queue slot for minutes |
| **Dead letter queue** | after `maxReceiveCount` deliveries | needs a human or a scheduled job to look at it, or it silently becomes a graveyard |
| **Metadata DB** | `{url, status, html_key, text_key}` | one more write per stage, but it is what makes the crawl resumable |

### What we actually measured

```
/ok/1      → fetched on delivery 1
/flaky/1   → 503, 503, fetched on delivery 3   ← backoff paid off
/broken/1  → 4 deliveries, then DLQ
bad DNS    → 4 deliveries, then DLQ
re-parse   → 0 additional origin requests
```

### The trade-off worth arguing about

Retrying is not free politeness-wise: a site that is down is often a site that is
*overloaded*, and hammering it with 4 retries makes it worse. Exponential backoff plus a
low `maxReceiveCount` is the compromise. Retry budgets per domain (stop retrying a domain
entirely once its error rate crosses a threshold) are the production-grade version.

**Next:** Lab 3 — Politeness (robots.txt + per-domain rate limiting)